# MatrixDB on Google Colab

Real CUDA backend of the MatrixDB engine: one GPU thread per query, `atomicAdd` Delta Log, reconcile kernel.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

This notebook writes its own source files (cells below), builds with `nvcc`, and runs. No uploads needed. Success ends with `reads=5000 writes=5000 delta_applied=5000` — asserts fire on any drop.

## 1. Confirm a GPU is attached

In [ ]:
!nvcc --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Write the source files

In [ ]:
%%writefile types.hpp
#pragma once

#include <cstdint>
#include <cstddef> // ponytail: spec wrote <csize_t> (does not exist); size_t lives in <cstddef>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Apple Silicon L1 cache lines are typically 128 bytes
    constexpr size_t MATRIX_CACHE_LINE = 128;
#else
    // Standard Intel/AMD x86_64 cache lines are 64 bytes
    constexpr size_t MATRIX_CACHE_LINE = 64;
#endif

/**
 * @brief Represents a single transaction query record.
 * Designed with descending structural member alignment to prevent internal compiler padding.
 * Aligned to target CPU cache line boundaries to eliminate false sharing.
 */
struct alignas(MATRIX_CACHE_LINE) DatabaseQuery {
    uint64_t query_id;
    uint64_t timestamp_us;
    uint64_t transaction_id;
    uint32_t opcode;
    uint32_t payload_size;
    uint8_t payload[32];

    static constexpr size_t data_footprint =
        sizeof(uint64_t) * 3 +
        sizeof(uint32_t) * 2 +
        sizeof(uint8_t) * 32;

    static constexpr size_t padding_needed =
        (MATRIX_CACHE_LINE > data_footprint) ? (MATRIX_CACHE_LINE - data_footprint) : 0;

    // Explicit structural padding to align with target L1 cache line size
    uint8_t padding[padding_needed > 0 ? padding_needed : 1];
};

static_assert((sizeof(DatabaseQuery) % MATRIX_CACHE_LINE) == 0,
    "DatabaseQuery structure violates cache-line alignment constraints.");

// Operational instruction carried in DatabaseQuery::opcode (spec: Read, Write, Scan).
enum MatrixOpcode : uint32_t {
    OP_READ  = 1,
    OP_WRITE = 2,
    OP_SCAN  = 3,
};

// Shared store + append-only Delta Log layout (spec §2.3 store, §4 mutation path).
// Defined here so the CPU engine and the CUDA kernel agree byte-for-byte on the layout.
struct Mutation { uint64_t key; uint64_t value; };

constexpr size_t MATRIX_STORE_SLOTS       = 4096;                        // power of two
constexpr size_t MATRIX_STORE_MASK        = MATRIX_STORE_SLOTS - 1;
constexpr size_t MATRIX_DELTA_LOG_CAPACITY = 8192;                       // >= max queries per batch
constexpr size_t MATRIX_DELTA_LOG_MASK    = MATRIX_DELTA_LOG_CAPACITY - 1;
constexpr size_t MATRIX_BATCH_MAX         = 512;                         // host-side max batch (matches main)


In [ ]:
%%writefile ring_buffer.hpp
#pragma once

#include "types.hpp"
#include <atomic>
#include <array>
#include <utility>

/**
 * @brief Ultra-low latency, lock-free, zero-allocation SPSC Ring Buffer.
 * Enforces power-of-two capacities to replace expensive modulo operations with bitwise AND masking.
 */
template <typename T, size_t Capacity>
class SPSCRingBuffer {
    static_assert((Capacity & (Capacity - 1)) == 0, "SPSCRingBuffer Capacity must be a power of two.");
    static_assert(Capacity >= 2, "SPSCRingBuffer Capacity must be at least two elements.");

public:
    SPSCRingBuffer()
        : head_(0)
        , tail_(0)
        , cached_head_(0)
        , cached_tail_(0) {}

    ~SPSCRingBuffer() = default;

    SPSCRingBuffer(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer& operator=(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer(SPSCRingBuffer&&) = delete;
    SPSCRingBuffer& operator=(SPSCRingBuffer&&) = delete;

    /**
     * @brief Emplaces a new item at the tail of the buffer. Only the Producer may call this method.
     */
    template <typename... Args>
    bool try_emplace(Args&&... args) {
        const size_t current_tail = tail_.load(std::memory_order_relaxed);

        // Check local cached head before loading the atomic head_ cursor from shared memory
        if (current_tail - cached_head_ >= Capacity) {
            cached_head_ = head_.load(std::memory_order_acquire);
            if (current_tail - cached_head_ >= Capacity) {
                return false; // Queue is genuinely full
            }
        }

        const size_t index = current_tail & MASK;
        buffer_[index] = T{std::forward<Args>(args)...};

        tail_.store(current_tail + 1, std::memory_order_release);
        return true;
    }

    /**
     * @brief Pops an item from the head of the buffer. Only the Consumer may call this method.
     */
    bool try_pop(T& value) {
        const size_t current_head = head_.load(std::memory_order_relaxed);

        // Check local cached tail before loading the atomic tail_ cursor from shared memory
        if (current_head >= cached_tail_) {
            cached_tail_ = tail_.load(std::memory_order_acquire);
            if (current_head >= cached_tail_) {
                return false; // Queue is genuinely empty
            }
        }

        const size_t index = current_head & MASK;
        value = std::move(buffer_[index]);

        head_.store(current_head + 1, std::memory_order_release);
        return true;
    }

    [[nodiscard]] bool empty() const noexcept {
        return head_.load(std::memory_order_relaxed) == tail_.load(std::memory_order_relaxed);
    }

    [[nodiscard]] size_t size() const noexcept {
        const size_t t = tail_.load(std::memory_order_relaxed);
        const size_t h = head_.load(std::memory_order_relaxed);
        return (t >= h) ? (t - h) : 0;
    }

private:
    static constexpr size_t MASK = Capacity - 1;

    // Isolate variables on separate cache lines to eliminate false sharing
    alignas(MATRIX_CACHE_LINE) std::array<T, Capacity> buffer_{};
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> head_;
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> tail_;

    // Thread-local shadow indices
    alignas(MATRIX_CACHE_LINE) size_t cached_head_;
    alignas(MATRIX_CACHE_LINE) size_t cached_tail_;
};


In [ ]:
%%writefile compute.hpp
#pragma once

#include "types.hpp"
#include <cstddef>

/**
 * @brief Pure virtual interface defining the compute engine's entry point.
 * Enables zero-copy, raw pointer processing over batch slices.
 *
 * Two implementations satisfy this contract and are selected at compile time:
 *   - CPUMockEngine (compute_mock.cpp)  — default, runs anywhere, no GPU needed
 *   - CUDAGPUEngine (compute_cuda.cuh)  — built with nvcc when MATRIX_USE_CUDA is set
 *
 * The counters below let the caller verify dispatch/commit correctness identically
 * on both backends (same asserts in main.cpp prove the same thing on CPU and GPU).
 */
class ComputeInterface {
public:
    virtual ~ComputeInterface() = default;
    virtual void execute_batch(DatabaseQuery* batch_array, size_t count) = 0;

    virtual uint64_t reads() const = 0;
    virtual uint64_t writes() const = 0;
    virtual uint64_t delta_applied() const = 0;
};


In [ ]:
%%writefile compute_mock.cpp
#include "compute.hpp"
#include <vector>
#include <thread>
#include <atomic>
#include <array>
#include <chrono>
#include <iostream>

/**
 * @brief CPU Mock Engine — the no-GPU fallback (Component 5: Local Sandbox).
 * Executes the real opcode-dispatch + Delta Log mechanics serially, simulating
 * GPU kernel-launch latency with a spin-wait. Same correctness contract as the
 * CUDA engine, so it is the runnable proof when no GPU is present.
 */
class CPUMockEngine : public ComputeInterface {
public:
    explicit CPUMockEngine(size_t worker_count)
        : shutdown_(false) {
        // ponytail: workers kept only to mirror the spec's pinned-thread-pool shape.
        // They do no work in the serial CPU mock; real parallelism lives in the CUDA engine.
        for (size_t i = 0; i < worker_count; ++i) {
            workers_.emplace_back([this, i]() {
                this->worker_loop(i);
            });
        }
        std::cout << "CPUMockEngine initialized with " << worker_count << " worker threads." << std::endl;
    }

    ~CPUMockEngine() override {
        shutdown_.store(true, std::memory_order_relaxed);
        for (auto& thread : workers_) {
            if (thread.joinable()) {
                thread.join();
            }
        }
        std::cout << "CPUMockEngine workers shutdown cleanly." << std::endl;
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        // Simulates query processing and kernel execution latency using a spin-wait loop
        const auto start_wait = std::chrono::high_resolution_clock::now();
        while (true) {
            const auto current_time = std::chrono::high_resolution_clock::now();
            const auto elapsed = std::chrono::duration_cast<std::chrono::microseconds>(current_time - start_wait).count();
            if (elapsed >= 50) { // Emulate a 50-microsecond GPU execution delay
                break;
            }
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
            asm volatile("isb" ::: "memory");
#else
            asm volatile("pause" ::: "memory");
#endif
        }

        // Execution phase: dispatch every query on its opcode.
        // Reads hit the columnar store directly. Writes never touch the store here —
        // they reserve a Delta Log slot with a wait-free atomic fetch-add and stage
        // the mutation, exactly as the spec's append-only log prescribes.
        for (size_t i = 0; i < count; ++i) {
            DatabaseQuery& q = batch_array[i];
            switch (q.opcode) {
                case OP_READ:
                    q.transaction_id = store_[q.query_id & MATRIX_STORE_MASK];
                    reads_.fetch_add(1, std::memory_order_relaxed);
                    break;
                case OP_WRITE: {
                    // ponytail: mask guards overflow; capacity already >= batch size so it never wraps in practice
                    const size_t slot = delta_head_.fetch_add(1, std::memory_order_relaxed) & MATRIX_DELTA_LOG_MASK;
                    delta_log_[slot] = Mutation{q.query_id, q.query_id};
                    writes_.fetch_add(1, std::memory_order_relaxed);
                    break;
                }
                default:
                    break; // OP_SCAN and unknown opcodes are no-ops in the mock
            }
        }

        // Reconciliation phase: commit staged mutations into the store, then reset the
        // log for the next batch. Single consumer ⇒ no validation needed yet.
        // ponytail: last-writer-wins. Full OCC (TEV lock bit + read-set validation) is the
        // upgrade path once batches are processed by concurrent workers.
        const size_t logged = delta_head_.load(std::memory_order_relaxed);
        for (size_t s = 0; s < logged; ++s) {
            const Mutation& m = delta_log_[s & MATRIX_DELTA_LOG_MASK];
            store_[m.key & MATRIX_STORE_MASK] = m.value;
            delta_applied_.fetch_add(1, std::memory_order_relaxed);
        }
        delta_head_.store(0, std::memory_order_relaxed);
    }

    uint64_t reads() const override { return reads_.load(std::memory_order_relaxed); }
    uint64_t writes() const override { return writes_.load(std::memory_order_relaxed); }
    uint64_t delta_applied() const override { return delta_applied_.load(std::memory_order_relaxed); }

private:
    void worker_loop(size_t thread_id) {
        (void)thread_id;
        while (!shutdown_.load(std::memory_order_relaxed)) {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
            asm volatile("isb" ::: "memory");
#else
            asm volatile("pause" ::: "memory");
#endif
        }
    }

    std::array<uint64_t, MATRIX_STORE_SLOTS> store_{};        // the Value column
    std::array<Mutation, MATRIX_DELTA_LOG_CAPACITY> delta_log_{};
    std::atomic<size_t> delta_head_{0};                      // wait-free slot reservation cursor
    std::atomic<uint64_t> reads_{0};
    std::atomic<uint64_t> writes_{0};
    std::atomic<uint64_t> delta_applied_{0};

    std::vector<std::thread> workers_;
    std::atomic<bool> shutdown_;
};


In [ ]:
%%writefile compute_cuda.cuh
#pragma once

// CUDA backend (Component 4: Parallel Engine + Component 5: Enterprise path).
// Compile on a GPU host (e.g. Google Colab) with:
//     nvcc -std=c++17 -O3 -x cu -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto
// One GPU thread per query executes the opcode dispatch; writes reserve Delta Log
// slots via atomicAdd (the spec's wait-free fetch-add on real silicon); a second
// kernel reconciles the log into the device-resident columnar store.

#include "compute.hpp"
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>

// Trust boundary: a failed CUDA call means the device state is unknown — never swallow it.
#define CUDA_CHECK(call)                                                            \
    do {                                                                           \
        cudaError_t _err = (call);                                                 \
        if (_err != cudaSuccess) {                                                 \
            std::fprintf(stderr, "CUDA error %s at %s:%d -> %s\n",                 \
                         #call, __FILE__, __LINE__, cudaGetErrorString(_err));     \
            std::abort();                                                          \
        }                                                                          \
    } while (0)

// --- Device kernels ---

// One thread per query. Reads hit the store; writes stage into the append-only
// Delta Log at an atomically-reserved slot. Identical mechanics to the CPU mock.
__global__ void matrix_execute_kernel(DatabaseQuery* batch, size_t count,
                                      const uint64_t* store, Mutation* delta_log,
                                      unsigned long long* delta_head,
                                      unsigned long long* reads,
                                      unsigned long long* writes) {
    const size_t i = static_cast<size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (i >= count) return;

    const uint32_t opcode = batch[i].opcode;
    const uint64_t key = batch[i].query_id;
    if (opcode == OP_READ) {
        batch[i].transaction_id = store[key & MATRIX_STORE_MASK];
        atomicAdd(reads, 1ULL);
    } else if (opcode == OP_WRITE) {
        const unsigned long long slot =
            atomicAdd(delta_head, 1ULL) & MATRIX_DELTA_LOG_MASK;
        delta_log[slot].key = key;
        delta_log[slot].value = key; // mock projection: value == key
        atomicAdd(writes, 1ULL);
    }
    // OP_SCAN / unknown: no-op, matching the CPU mock.
}

// One thread per logged mutation. Commits the Delta Log into the store.
// ponytail: parallel last-writer-wins — on colliding keys the winner is
// nondeterministic. Test keys are unique so it's exact; colliding keys need
// the spec's §4 OCC validation (TEV lock bit + read-set check). Upgrade path.
__global__ void matrix_reconcile_kernel(uint64_t* store, const Mutation* delta_log,
                                        unsigned long long logged,
                                        unsigned long long* delta_applied) {
    const size_t s = static_cast<size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (s >= logged) return;
    const Mutation m = delta_log[s & MATRIX_DELTA_LOG_MASK];
    store[m.key & MATRIX_STORE_MASK] = m.value;
    atomicAdd(delta_applied, 1ULL);
}

/**
 * @brief Real CUDA GPU engine. Device-resident store + Delta Log persist across
 * batches (it is the database). Honors the same ComputeInterface contract and the
 * same correctness asserts as CPUMockEngine.
 */
class CUDAGPUEngine : public ComputeInterface {
public:
    explicit CUDAGPUEngine(size_t /*worker_count*/ = 0) {
        int device_count = 0;
        CUDA_CHECK(cudaGetDeviceCount(&device_count));
        if (device_count == 0) {
            std::fprintf(stderr, "CUDAGPUEngine: no CUDA device found.\n");
            std::abort();
        }
        cudaDeviceProp prop{};
        CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));

        CUDA_CHECK(cudaMalloc(&d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMemset(d_store_, 0, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMalloc(&d_delta_log_, MATRIX_DELTA_LOG_CAPACITY * sizeof(Mutation)));
        CUDA_CHECK(cudaMalloc(&d_delta_head_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_delta_head_, 0, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMalloc(&d_batch_, MATRIX_BATCH_MAX * sizeof(DatabaseQuery)));
        CUDA_CHECK(cudaMalloc(&d_reads_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMalloc(&d_writes_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMalloc(&d_delta_applied_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_reads_, 0, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_writes_, 0, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_delta_applied_, 0, sizeof(unsigned long long)));

        // ponytail: single stream. Hyper-Q multi-stream is a throughput upgrade
        // (spec §3) — add a stream pool when batches overlap; not needed to run.
        CUDA_CHECK(cudaStreamCreate(&stream_));

        std::printf("CUDAGPUEngine initialized on '%s' (%d SMs).\n",
                    prop.name, prop.multiProcessorCount);
    }

    ~CUDAGPUEngine() override {
        cudaFree(d_store_);
        cudaFree(d_delta_log_);
        cudaFree(d_delta_head_);
        cudaFree(d_batch_);
        cudaFree(d_reads_);
        cudaFree(d_writes_);
        cudaFree(d_delta_applied_);
        cudaStreamDestroy(stream_);
        std::printf("CUDAGPUEngine released device resources.\n");
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX; // ponytail: clamp; main never exceeds this

        // Stage batch to device. ponytail: plain async copy. cudaHostRegister to pin
        // the host pool for zero-copy DMA (spec §3) is the bandwidth upgrade.
        CUDA_CHECK(cudaMemcpyAsync(d_batch_, batch_array, count * sizeof(DatabaseQuery),
                                   cudaMemcpyHostToDevice, stream_));

        constexpr int TPB = 256;
        const int exec_blocks = static_cast<int>((count + TPB - 1) / TPB);
        matrix_execute_kernel<<<exec_blocks, TPB, 0, stream_>>>(
            d_batch_, count, d_store_, d_delta_log_, d_delta_head_, d_reads_, d_writes_);
        CUDA_CHECK(cudaGetLastError());

        // Read how many mutations were staged this batch to size the reconcile launch.
        unsigned long long logged = 0;
        CUDA_CHECK(cudaMemcpyAsync(&logged, d_delta_head_, sizeof(logged),
                                   cudaMemcpyDeviceToHost, stream_));
        CUDA_CHECK(cudaStreamSynchronize(stream_));

        if (logged > 0) {
            const int rec_blocks = static_cast<int>((logged + TPB - 1) / TPB);
            matrix_reconcile_kernel<<<rec_blocks, TPB, 0, stream_>>>(
                d_store_, d_delta_log_, logged, d_delta_applied_);
            CUDA_CHECK(cudaGetLastError());
        }

        // Reset the Delta Log head for the next batch, then copy read results back.
        CUDA_CHECK(cudaMemsetAsync(d_delta_head_, 0, sizeof(unsigned long long), stream_));
        CUDA_CHECK(cudaMemcpyAsync(batch_array, d_batch_, count * sizeof(DatabaseQuery),
                                   cudaMemcpyDeviceToHost, stream_));
        CUDA_CHECK(cudaStreamSynchronize(stream_));
    }

    uint64_t reads() const override { return read_counter(d_reads_); }
    uint64_t writes() const override { return read_counter(d_writes_); }
    uint64_t delta_applied() const override { return read_counter(d_delta_applied_); }

private:
    static uint64_t read_counter(const unsigned long long* d_ptr) {
        unsigned long long h = 0;
        CUDA_CHECK(cudaMemcpy(&h, d_ptr, sizeof(h), cudaMemcpyDeviceToHost));
        return static_cast<uint64_t>(h);
    }

    uint64_t* d_store_ = nullptr;
    Mutation* d_delta_log_ = nullptr;
    unsigned long long* d_delta_head_ = nullptr;
    DatabaseQuery* d_batch_ = nullptr;
    unsigned long long* d_reads_ = nullptr;
    unsigned long long* d_writes_ = nullptr;
    unsigned long long* d_delta_applied_ = nullptr;
    cudaStream_t stream_{};
};


In [ ]:
%%writefile main.cpp
#include "types.hpp"
#include "ring_buffer.hpp"
#if defined(MATRIX_USE_CUDA)
    #include "compute_cuda.cuh"   // real GPU engine (build with nvcc -DMATRIX_USE_CUDA)
    using EngineType = CUDAGPUEngine;
#else
    #include "compute_mock.cpp"   // CPU fallback (default; runs anywhere)
    using EngineType = CPUMockEngine;
#endif
#include <iostream>
#include <chrono>
#include <thread>
#include <memory>
#include <array>
#include <atomic>
#include <cassert>
#include <vector>
#include <algorithm>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    #include <pthread.h>
    #include <pthread/qos.h>        // ponytail: spec omitted these 3; needed to compile the
    #include <mach/mach.h>          // QoS + Mach affinity calls below on macOS
    #include <mach/thread_policy.h>
#elif defined(__linux__)
    #include <sched.h>
#endif

/**
 * @brief Platform-agnostic pipeline stalling barrier for low-overhead busy spins.
 */
inline void spin_stall() noexcept {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Instruction Synchronization Barrier (isb) forces instruction execution pipeline flush on ARM64
    asm volatile("isb" ::: "memory");
#elif defined(__x86_64__)
    // Standard pause execution hint
    asm volatile("pause" ::: "memory");
#else
    asm volatile("" ::: "memory");
#endif
}

/**
 * @brief Configures OS-specific thread-pinning and priority policies to ensure execution on performance cores.
 */
inline void pin_to_performance_core() {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Promotes the calling thread to the interactive QoS class to guarantee performance core placement on macOS
    pthread_set_qos_class_self_np(QOS_CLASS_USER_INTERACTIVE, 0);

    // Advisory affinity grouping configuration on Apple Silicon Mach kernels
    thread_port_t mach_thread = pthread_mach_thread_np(pthread_self());
    thread_affinity_policy_data_t policy = { 1 };
    thread_policy_set(mach_thread, THREAD_AFFINITY_POLICY, (thread_policy_t)&policy, THREAD_AFFINITY_POLICY_COUNT);
#elif defined(__linux__)
    cpu_set_t cpuset;
    CPU_ZERO(&cpuset);
    CPU_SET(0, &cpuset); // Pin strictly to logical Core 0
    pthread_setaffinity_np(pthread_self(), sizeof(cpu_set_t), &cpuset);
#endif
}

int main() {
    std::cout << "MatrixDB Bare-Metal Engine Booting..." << std::endl;

    constexpr size_t BATCH_SIZE_LIMIT = 512;
    constexpr uint64_t TEMPORAL_LIMIT_US = 50;
    constexpr size_t TOTAL_QUERIES = 10000;

    auto ring_buffer = std::make_unique<SPSCRingBuffer<DatabaseQuery, 4096>>();
    auto mock_engine = std::make_unique<EngineType>(4);

    std::atomic<bool> run_state{true};
    std::atomic<size_t> total_processed{0}; // ponytail: end-to-end check that every query flows through

    // Pre-reserved so the hot path never allocates. Filled by the single consumer only.
    std::vector<uint64_t> latencies_ns;
    latencies_ns.reserve(TOTAL_QUERIES);

    // Spin up the consumer thread to execute the ingestion busy-spin control loop
    std::thread consumer([&]() {
        pin_to_performance_core();

        std::array<DatabaseQuery, BATCH_SIZE_LIMIT> batch;
        size_t accumulated_queries = 0;
        auto batch_start_time = std::chrono::high_resolution_clock::now();

        while (run_state.load(std::memory_order_relaxed)) {
            DatabaseQuery incoming_query;
            const bool success = ring_buffer->try_pop(incoming_query);

            if (success) {
                // Queue residency = time from producer's enqueue stamp to this pop.
                const uint64_t now_ns = static_cast<uint64_t>(
                    std::chrono::steady_clock::now().time_since_epoch().count());
                latencies_ns.push_back(now_ns - incoming_query.timestamp_us);

                if (accumulated_queries == 0) {
                    batch_start_time = std::chrono::high_resolution_clock::now();
                }
                batch[accumulated_queries++] = incoming_query;
            }

            // Continuous evaluation of the Dual-Trigger Condition
            if (accumulated_queries > 0) {
                const auto current_time = std::chrono::high_resolution_clock::now();
                const auto duration_us = std::chrono::duration_cast<std::chrono::microseconds>(
                    current_time - batch_start_time
                ).count();

                // Trigger flush if batch is full OR time-based window has expired
                if (accumulated_queries >= BATCH_SIZE_LIMIT || duration_us >= static_cast<long long>(TEMPORAL_LIMIT_US)) {
                    mock_engine->execute_batch(batch.data(), accumulated_queries);
                    total_processed.fetch_add(accumulated_queries, std::memory_order_relaxed);
                    accumulated_queries = 0;
                }
            }

            if (!success) {
                spin_stall();
            }
        }
    });

    // Simulate query production on a separate thread
    std::thread producer([&]() {
        for (size_t i = 0; i < TOTAL_QUERIES; ++i) {
            // Stamp ingestion time (steady_clock ns) so the consumer can measure queue residency.
            const uint64_t stamp_ns = static_cast<uint64_t>(
                std::chrono::steady_clock::now().time_since_epoch().count());
            // Alternate Read/Write so both opcode paths are exercised end to end.
            const uint32_t op = (i % 2 == 0) ? OP_READ : OP_WRITE;
            DatabaseQuery q{i, stamp_ns, 0, op, 8, {0}, {0}};
            while (!ring_buffer->try_emplace(q)) {
                spin_stall();
            }
        }
    });

    producer.join();

    // Allow the queue to drain before stopping
    std::this_thread::sleep_for(std::chrono::milliseconds(50));
    run_state.store(false);

    if (consumer.joinable()) {
        consumer.join();
    }

    const size_t processed = total_processed.load();
    std::cout << "Processed " << processed << " / " << TOTAL_QUERIES << " queries." << std::endl;
    assert(processed == TOTAL_QUERIES && "Dual-trigger pipeline dropped queries");

    // Verify the engine actually dispatched on opcode and committed every mutation.
    const uint64_t reads = mock_engine->reads();
    const uint64_t writes = mock_engine->writes();
    const uint64_t applied = mock_engine->delta_applied();
    std::cout << "Engine: reads=" << reads << " writes=" << writes
              << " delta_applied=" << applied << std::endl;
    assert(reads + writes == processed && "opcode dispatch missed queries");
    assert(applied == writes && "delta log reconcile dropped mutations");

    // Report SPSC queue-residency latency distribution (the spec's core thesis).
    if (!latencies_ns.empty()) {
        std::sort(latencies_ns.begin(), latencies_ns.end());
        const auto pct = [&](double p) {
            const size_t idx = static_cast<size_t>(p * (latencies_ns.size() - 1));
            return latencies_ns[idx];
        };
        std::cout << "SPSC queue latency (ns)  "
                  << "p50=" << pct(0.50) << "  "
                  << "p99=" << pct(0.99) << "  "
                  << "p99.9=" << pct(0.999) << "  "
                  << "max=" << latencies_ns.back() << std::endl;
    }

    std::cout << "Engine execution loop completed successfully." << std::endl;
    return 0;
}


## 3. Build & run on the GPU

`-x cu` compiles `main.cpp` as CUDA so the `.cuh` kernels link in. `-D_GNU_SOURCE` exposes Linux thread-affinity APIs; `-Xcompiler -pthread` links std::thread.

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto

In [ ]:
!./matrixdb_proto

## 4. CPU fallback (run this if no GPU runtime is selected)

Same code, same asserts, no CUDA — proves the logic without a GPU.

In [ ]:
!g++ -std=c++17 -O3 -D_GNU_SOURCE -pthread main.cpp -o matrixdb_cpu && ./matrixdb_cpu